In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

# Hands-On Lab: Building Agent Systems with Databricks

## Part 2 - Agent Evaluation
Now that we've created an agent, how do we evaluate its performance?
For the second part, we're going to create a product support agent so we can focus on evaluation.
This agent will use a RAG approach to help answer questions about products using the product documentation.

### 2.1 Define our new Agent and retriever tool
- [**agent.py**]($./agent.py): An example Agent has been configured - first we'll explore this file and understand the building blocks
- **Vector Search**: We've created a Vector Search endpoint that can be queried to find related documentation about a specific product.
- **Create Retriever Function**: Define some properties about our retriever and package it so it can be called by our LLM.

### 2.2 Create Evaluation Dataset
- We've provided an example evaluation dataset - though you can also generate this [synthetically](https://www.databricks.com/blog/streamline-ai-agent-evaluation-with-new-synthetic-data-capabilities).

### 2.3 Run MLflow.genai.evaluate() 
- MLflow will take your evaluation dataset and test your agent's responses against it
- LLM Judges will score the outputs and collect everything in a nice UI for review

### 2.4 Make Needed Improvements and re-run Evaluations
- Take feedback from our evaluation run and change the prompt
- Run evals again and see the improvement!

In [0]:
%pip install -U -qqqq backoff databricks-openai uv databricks-agents mlflow-skinny[databricks] unitycatalog-langchain[databricks] databricks-langchain

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jupyter-server 1.23.4 requires anyio<4,>=3.1.0, but you have anyio 4.12.1 which is incompatible.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
# --- Lab hygiene: suppress known non-actionable warnings ---
import warnings, logging

# Pydantic v2 serializer warnings (safe to ignore for this lab)
warnings.filterwarnings(
    "ignore",
    message=r"^Pydantic serializer warnings:",
    category=UserWarning,
    module=r"pydantic\..*",
)

# MLflow tracing warnings (otel not fully enabled in this runtime)
logging.getLogger("mlflow.tracing").setLevel(logging.ERROR)
logging.getLogger("mlflow.tracing.fluent").setLevel(logging.ERROR)

In [0]:
%reload_ext autoreload

In [0]:
from agent import AGENT

AGENT.predict({"input": [{"role": "user", "content": "Hello, what do you do?"}]})

/local_disk0/.ephemeral_nfs/envs/pythonEnv-0253fb5b-5ee4-46db-a6d5-4264758f16ab/lib/python3.10/site-packages/databricks/connect/session.py:451: UserWarning: Ignoring the default notebook Spark session and creating a new Spark Connect session. To use the default notebook Spark session, use DatabricksSession.builder.getOrCreate() with no additional parameters.
  warnings.warn(new_notebook_session_msg)


ResponsesAgentResponse(tool_choice=None, truncation=None, id=None, created_at=None, error=None, incomplete_details=None, instructions=None, metadata=None, model=None, object='response', output=[OutputItem(type='reasoning', summary=[{'type': 'summary_text', 'text': 'We need to introduce role and capabilities. The user asks "Hello, what do you do?" We should respond as customer success specialist for Databricks lab, explain we help with product questions, can retrieve info, etc. No tool needed.'}], id='chatcmpl_2adb97e3-9a4d-4c3d-a05b-c260826e11ae'), OutputItem(type='message', id='chatcmpl_2adb97e3-9a4d-4c3d-a05b-c260826e11ae', content=[{'text': 'Hi there! I’m a Customer Success Specialist for the Databricks Labs experience. My role is to help you get the most out of Databricks’ products and services—whether you’re looking for technical guidance, need clarification on features, want help with troubleshooting, or have questions about policies (like returns or licensing). \n\nI can:\n\n* A

Trace(trace_id=tr-911d25236c7d9c98ed8268476e430307)

### Log the `agent` as an MLflow model
Log the agent as code from the [agent]($./agent) notebook. See [MLflow - Models from Code](https://mlflow.org/docs/latest/models.html#models-from-code).

In [0]:
# Determine Databricks resources to specify for automatic auth passthrough at deployment time
import mlflow
from agent import VECTOR_SEARCH_TOOLS, LLM_ENDPOINT_NAME
from databricks_openai import UCFunctionToolkit, VectorSearchRetrieverTool
from mlflow.models.resources import DatabricksFunction, DatabricksServingEndpoint
from unitycatalog.ai.langchain.toolkit import UnityCatalogTool


resources = [DatabricksServingEndpoint(endpoint_name=LLM_ENDPOINT_NAME)]
for tool in VECTOR_SEARCH_TOOLS:
    if isinstance(tool, VectorSearchRetrieverTool):
        resources.extend(tool.resources)
    elif isinstance(tool, UnityCatalogTool):
        resources.append(DatabricksFunction(function_name=tool.uc_function_name))

input_example = {
    "input": [
        {"role": "user", "content": "What color options are available for the Aria Modern Bookshelf?"}
    ],
}

with mlflow.start_run():
    logged_agent_info = mlflow.pyfunc.log_model(
        artifact_path="agent",
        python_model="agent.py",  
        input_example=input_example,
        resources=resources,
        extra_pip_requirements=[
            "mlflow>=3.1.3",
            "databricks-agents>=1.1.0",
            "databricks-openai",
        ],
    )

2026/01/21 16:50:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-dd7874a8-6e13.cloud.databricks.com/ml/experiments/89e7cdeb0a8e4041a2e33e3606bd0ec6/models/m-589584dd3e504a95a59188d870fff91f?o=2703287350484668
2026/01/21 16:50:07 INFO mlflow.pyfunc: Predicting on input example to validate output
2026/01/21 16:50:07 WARNING mlflow.tracing.fluent: Failed to start span predict_stream: 'NonRecordingSpan' object has no attribute 'context'. For full traceback, set logging level to debug.
2026/01/21 16:50:07 WARNING mlflow.tracing.fluent: Failed to start span Completions: 'NonRecordingSpan' object has no attribute 'context'. For full traceback, set logging level to debug.
2026/01/21 16:50:08 WARNING mlflow.tracing.fluent: Failed to start span Completions: 'NonRecordingSpan' object has no attribute 'context'. For full traceback, set logging level to debug.
2026/01/21 16:50:09 WARNING mlflow.tracing.fluent: Failed to s

In [0]:
# Load the model and create a prediction function
logged_model_uri = f"runs:/{logged_agent_info.run_id}/agent"
loaded_model = mlflow.pyfunc.load_model(logged_model_uri)

def predict_wrapper(query):
    model_input = {
        "input": [
            {"role": "user", "content": query}
        ]
    }
    response = loaded_model.predict(model_input)
    # Find the last output item of type "message"
    message = next(
        (item for item in reversed(response["output"]) if item.get("type") == "message"),
        None
    )
    if message and "content" in message:
        # Find the first content item of type "output_text"
        content_item = next(
            (c for c in message["content"] if c.get("type") == "output_text"),
            None
        )
        if content_item:
            return content_item["text"]
    return None

## Evaluate the agent with [Agent Evaluation](https://docs.databricks.com/generative-ai/agent-evaluation/index.html)

You can edit the requests or expected responses in your evaluation dataset and run evaluation as you iterate your agent, leveraging mlflow to track the computed quality metrics.

In [0]:
import pandas as pd

data = {
    "request": [
        "What are my bill payment options?"
    ],
    "expected_facts": 
        [
            "Payments can be done through online payment portal."
            "Payments can be done through mobile app."
            "Payments can be done through in-store payment."
            "Payments can be done through mail."
            "Payments can be done through phone payment."
    ]
}

eval_dataset = pd.DataFrame(data)

In [0]:
from mlflow.genai.scorers import RetrievalGroundedness, RelevanceToQuery, Safety, Guidelines
import mlflow.genai

eval_data = []
for request, facts in zip(data["request"], data["expected_facts"]):
    eval_data.append({
        "inputs": {
            "query": request  # This matches the function parameter
        },
        "expected_response": "\n".join(facts)
    })

# Define custom scorers tailored to product information evaluation
scorers = [
    #RetrievalGroundedness(),  # Pre-defined judge that checks against retrieval results
    RelevanceToQuery(),  # Checks if answer is relevant to the question
    Safety(),  # Checks for harmful or inappropriate content
    Guidelines(
        guidelines="""Response must be clear and direct:
        - Answers the exact question asked
        - Uses lists for options, steps for instructions
        - No marketing fluff or extra background
        - Does not tell user to contact customer support
        - Concise but complete.""",
        name="clarity_and_structure",
    ),
   
]

In [0]:
print("Running evaluation...")
with mlflow.start_run():
    results = mlflow.genai.evaluate(
        data=eval_data,
        predict_fn=predict_wrapper, 
        scorers=scorers,
    )

Running evaluation...


2026/01/21 16:55:38 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2026/01/21 16:55:38 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.
2026/01/21 16:55:38 WARNING mlflow.tracing.fluent: Failed to start span predict_stream: 'NonRecordingSpan' object has no attribute 'context'. For full traceback, set logging level to debug.


Evaluating:   0%|          | 0/1 [Elapsed: 00:00, Remaining: ?] 

## Lets go back to the [agent.py]($./agent.py) file and change our prompt to better fit how we'd like it to respond and re-evaluate.